In [20]:
import prody as pr
import seaborn as sns
import matplotlib.pyplot as plt

In [21]:
n_replicas = 10

pdb_inicial = 'init_ca.pdb'
pdb_final = './inputs/4ake_target.pdb'
experimental_dcd = './experimental.dcd'
replica_dcd_template = "./results/{i}_ca.dcd"
output_fig = "mdff_nm.png"

In [22]:
# it fails with a clear message if the file does not exist
def load_pdb(path, **kwargs):
    structure = pr.parsePDB(path, **kwargs)
    if structure is None:
        raise FileNotFoundError(f"Could not load the PDB: {path}")
    return structure

In [23]:
def load_dcd(path):
    traj = pr.parseDCD(path)
    if traj is None:
        raise FileNotFoundError(f"Could not load the DCD: {path}")
    return traj

In [24]:
def project_onto_pca(name, coordsets, reference_ensemble, pca_modes):
    ens = pr.Ensemble(name)
    ens.setAtoms(reference_ensemble.getAtoms())
    ens.setCoords(reference_ensemble.getCoords())
    ens.addCoordset(coordsets)
    ens.superpose()
    proj = pr.calcProjection(ens, pca_modes, rmsd=False)
    return ens, proj

In [25]:
# initial/final structures and experimental trajectory
q_inicial = load_pdb(pdb_inicial, subset="calpha", compressed=False)
q_final = load_pdb(pdb_final, subset="calpha", compressed=False)
 
tmp_exp = load_dcd(experimental_dcd)
tmp_exp.setAtoms(q_inicial)

@> DCD file contains 24 coordinate sets for 214 atoms.
@> DCD file was parsed in 0.00 seconds.
@> 0.06 MB parsed at input rate 49.64 MB/s.
@> 24 coordinate sets parsed at input rate 19656 frame/s.


In [26]:
# experimental ensemble + PCA
ens_exp = pr.PDBEnsemble(tmp_exp)
ens_exp.addCoordset(tmp_exp.getCoordsets())
ens_exp.iterpose()

pca_exp = pr.PCA("experimental")
pca_exp.buildCovariance(ens_exp)
pca_exp.calcModes()

exp_proj_2d = pr.calcProjection(ens_exp, pca_exp[:2], rmsd=False)

@> Starting iterative superposition:
@> Step #1: RMSD difference = 4.3960e+00
@> Step #2: RMSD difference = 4.0941e-03
@> Step #3: RMSD difference = 1.3260e-04
@> Step #4: RMSD difference = 4.6370e-06
@> Final superposition to calculate transformations.
@> Covariance is calculated using 48 coordinate sets.


In [27]:
# MDFF trajectories with Normal Modes (one per replicate)
mdff_proj_2d = {}
mdff_traj = {}
 
for i in range(1, n_replicas + 1):
    traj = load_dcd(replica_dcd_template.format(i=i))
    traj.setAtoms(q_inicial)
 
    _, proj = project_onto_pca(
        f"Replica_{i}", traj.getCoordsets(), ens_exp, pca_exp[:2]
    )
 
    mdff_traj[i] = traj
    mdff_proj_2d[i] = proj

@> DCD file contains 40 coordinate sets for 214 atoms.
@> DCD file was parsed in 0.00 seconds.
@> 0.10 MB parsed at input rate 50.16 MB/s.
@> 40 coordinate sets parsed at input rate 19864 frame/s.
@> DCD file contains 40 coordinate sets for 214 atoms.
@> DCD file was parsed in 0.00 seconds.
@> 0.10 MB parsed at input rate 194.89 MB/s.
@> 40 coordinate sets parsed at input rate 77172 frame/s.
@> DCD file contains 40 coordinate sets for 214 atoms.
@> DCD file was parsed in 0.00 seconds.
@> 0.10 MB parsed at input rate 158.21 MB/s.
@> 40 coordinate sets parsed at input rate 62648 frame/s.
@> DCD file contains 40 coordinate sets for 214 atoms.
@> DCD file was parsed in 0.00 seconds.
@> 0.10 MB parsed at input rate 171.60 MB/s.
@> 40 coordinate sets parsed at input rate 67951 frame/s.
@> DCD file contains 40 coordinate sets for 214 atoms.
@> DCD file was parsed in 0.00 seconds.
@> 0.10 MB parsed at input rate 163.65 MB/s.
@> 40 coordinate sets parsed at input rate 64801 frame/s.
@> DCD file

In [28]:
# projection of the initial and final structures (crystallographic)
tmp_inicial = load_pdb(pdb_inicial, subset="calpha")
ens_inicial, cristal_inicial_proj_2d = project_onto_pca(
    "Inicial", tmp_inicial.getCoords(), ens_exp, pca_exp[:2])

tmp_final = load_pdb(pdb_final, subset="calpha")
ens_final, cristal_final_proj_2d = project_onto_pca(
    "Final", tmp_final.getCoords(), ens_exp, pca_exp[:2])

In [29]:
# merge all replicates into a single trajectory/ensemble
tmp_total = mdff_traj[1]
for i in range(2, n_replicas + 1):
    tmp_total += mdff_traj[i]
 
tmp_total.setAtoms(q_inicial) 
ens_mdff_total, mdfftotal_proj_2d = project_onto_pca(
    "MDFF_total", tmp_total.getCoordsets(), ens_exp, pca_exp[:2])
 
print(ens_mdff_total.numCoordsets(), "coordsets")
print(ens_mdff_total.numAtoms(), "atoms")

800 coordsets
214 atoms


In [30]:
# axis limits
x_pad, y_pad = 118.0, 20.0
x_lim = (-10 - x_pad, 2 + x_pad)
y_lim = (-1 - y_pad, 2 + y_pad)

with plt.style.context({
    "figure.figsize": (20, 10),
    "figure.dpi": 600,
    "axes.labelsize": 15,
    "legend.fontsize": 15,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
}):

    fig, ax = plt.subplots(nrows=1, ncols=2)

# Graph 1

    sns.kdeplot(
        x=mdfftotal_proj_2d[:, 0],
        y=mdfftotal_proj_2d[:, 1],
        levels=20,
        cmap=plt.cm.Reds,
        fill=True,
        ax=ax[0],
    )

    for i in range(1, n_replicas + 1):
        ax[0].scatter(
            *mdff_proj_2d[i].T,
            c="black",
            s=30,
            marker="o",
            label="MDFF+NM" if i == 1 else None,
        )

    ax[0].scatter(
        *cristal_final_proj_2d[0].T,
        c="yellow",
        s=400,
        marker="d",
        label="Final",
    )

    ax[0].scatter(
        *cristal_inicial_proj_2d[0].T,
        c="green",
        s=400,
        marker="^",
        label="Initial",
    )

    ax[0].scatter(
        *exp_proj_2d.T,
        c="blue",
        s=80,
        marker="s",
        label="Experimental Structures",
    )

    ax[0].set_xlim(*x_lim)
    ax[0].set_ylim(*y_lim)
    ax[0].set_xlabel("PC1")
    ax[0].set_ylabel("PC2")
    ax[0].set_title("MDFF + Normal Modes")
    ax[0].legend()

# Graph 2
    sns.kdeplot(
        x=mdfftotal_proj_2d[:, 0],
        y=mdfftotal_proj_2d[:, 1],
        levels=20,
        cmap=plt.cm.Reds,
        fill=True,
        ax=ax[1],
    )

    for i in range(1, n_replicas + 1):

        ax[1].scatter(
            *mdff_proj_2d[i].T,
            c="black",
            s=30,
            marker="o",
            label="MDFF+NM" if i == 1 else None,
        )

        ax[1].plot(
            *mdff_proj_2d[i].T,
            linestyle="dashed",
            color="black",
        )

    ax[1].scatter(
        *cristal_final_proj_2d[0].T,
        c="yellow",
        s=400,
        marker="d",
        label="Final",
    )

    ax[1].scatter(
        *cristal_inicial_proj_2d[0].T,
        c="green",
        s=400,
        marker="^",
        label="Initial",
    )

    ax[1].scatter(
        *exp_proj_2d.T,
        c="blue",
        s=80,
        marker="s",
        label="Experimental Structures",
    )

    ax[1].set_xlim(*x_lim)
    ax[1].set_ylim(*y_lim)
    ax[1].set_xlabel("PC1")
    ax[1].set_ylabel("PC2")
    ax[1].set_title("MDFF + Normal Modes")
    ax[1].legend()

    plt.tight_layout()
    plt.savefig(output_fig, dpi=600)
    plt.show()